# Deep Learning Project Experiments

Based on the keras tutorial: [Image classification from scratch](https://keras.io/examples/vision/image_classification_from_scratch/)

## Imports

In [1]:
import os
import numpy as np
import keras
from keras import layers
from tensorflow import data as tf_data
import pydot
from keras import Sequential

2026-03-21 17:14:29.896173: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774113269.922301    1974 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774113269.932214    1974 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774113270.025578    1974 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774113270.025613    1974 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774113270.025616    1974 computation_placer.cc:177] computation placer alr

In [2]:
# Enable mixed_precision to save VRAM
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

In [3]:
import tensorflow as tf

# Check for GPU devices
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"Success! {len(gpus)} GPU(s) detected: {gpus}")
    try:
        # Currently, memory growth needs to be the same across GPUs
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Memory growth enabled!")
    except RuntimeError as e:
        # Memory growth must be set before GPUs have been initialized
        print(e)
else:
    print("No GPU detected. TensorFlow will fall back to the CPU.")

Success! 1 GPU(s) detected: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Memory growth enabled!


## Importing data

In [4]:
n_classes = 23
batch_size = 4
input_shape = (224, 224, 3)
image_size = (224, 224)
value_range = (0.0, 1.0)

train_ds, val_ds = keras.utils.image_dataset_from_directory(
    "wikiart_datasets",
    label_mode="categorical",
    validation_split=0.2,
    subset="both",
    seed=1337,
    image_size=image_size,
    batch_size=batch_size,
)

Found 13340 files belonging to 23 classes.
Using 10672 files for training.
Using 2668 files for validation.


I0000 00:00:1774113276.613825    1974 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1751 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Ti Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


In [5]:
# plt.figure(figsize=(10, 10))
# for images, labels in train_ds.take(1):
#     for i in range(9):
#         ax = plt.subplot(3, 3, i + 1)
#         plt.imshow(np.array(images[i]).astype("uint8"))
#         # plt.title(int(labels[i]))
#         plt.axis("off")

## Using image data augmentation

In [6]:
# data_augmentation_layers = [
#     layers.RandomFlip("horizontal"),
#     layers.RandomRotation(0.1),
# ]

# def data_augmentation(images):
#     for layer in data_augmentation_layers:
#         images = layer(images)
#     return images


We will apply the data augmentation as a part of the model so that we can benefit from GPU acceleration.

## Standardizing data

Our images are already in a standard size (512x512), however the RGB values are in the [0, 255] range, which is not ideal for a neural network.

So let's standardize the values to be in the [0, 1] range using Rescaling layer at the start of our model.

## Configure the dataset for performance

In [7]:
# # Prefetching samples in GPU memory helps maximize GPU utilization.
# train_ds = train_ds.prefetch(tf_data.AUTOTUNE)
# val_ds = val_ds.prefetch(tf_data.AUTOTUNE)

## Build a model

In [8]:
# python standard library imports
from typing import Self, Any
from pathlib import Path

In [9]:
# augmentation operations
from keras.layers import Rescaling, RandAugment

In [10]:
# model building
from keras import Model
from keras.applications import VGG16
from keras.layers import Flatten, Dense

In [11]:
# model training imports
from keras.optimizers import SGD
from keras.losses import CategoricalCrossentropy
from keras.metrics import CategoricalAccuracy, AUC, F1Score
from keras.callbacks import ModelCheckpoint, CSVLogger, LearningRateScheduler

In [12]:
def exp_decay_lr_scheduler(
    epoch: int,
    current_lr: float,
    factor: float = 0.95
) -> float:
    """
    Exponential decay learning rate scheduler
    """

    current_lr *= factor

    return current_lr

In [13]:
class AugmentedVGG16(Model):
    """
    Pre-trained VG16 + RandAugment
    """

    def __init__(self: Self) -> None:
        """
        Initialization
        """

        super().__init__()

        self.n_classes = n_classes
        self.rescale_layer = Rescaling(scale=1 / 255.0)
        self.augmentation_layer = RandAugment(value_range=value_range)
        self.pre_trained_architecture = VGG16(include_top=False, classes=n_classes)
        # self.pre_trained_architecture.trainable = False
        self.flatten_layer = Flatten()
        self.dense_layer = Dense(self.n_classes, activation="softmax")

    def call(self: Self, inputs: Any) -> Any:
        """
        Forward call
        """

        x = self.rescale_layer(inputs)
        x = self.augmentation_layer(x)
        x = self.pre_trained_architecture(x)
        x = self.flatten_layer(x)

        return self.dense_layer(x)

In [14]:
epochs = 4
model = AugmentedVGG16()
optimizer = SGD(learning_rate=0.001, name="optimizer")
loss = CategoricalCrossentropy(name="loss")

In [15]:
# metrics
categorical_accuracy = CategoricalAccuracy(name="accuracy")
auc = AUC(name="auc")
f1_score = F1Score(average="macro", name="f1_score")
metrics = [categorical_accuracy, auc, f1_score]

In [16]:
# callbacks
root_dir_path = Path(".")
checkpoint_file_path = root_dir_path / "checkpoint.keras"
metrics_file_path = root_dir_path / "metrics.csv"

checkpoint_callback = ModelCheckpoint(
    checkpoint_file_path,
    monitor="val_loss",
    verbose=0
)
metrics_callback = CSVLogger(metrics_file_path)
lr_scheduler_callback = LearningRateScheduler(exp_decay_lr_scheduler)

callbacks = [
    checkpoint_callback,
    metrics_callback,
    lr_scheduler_callback
]

In [17]:
model.compile(loss=loss, optimizer=optimizer, metrics=metrics)

In [18]:
# train the model, call to the method is somewhat diferent
_ = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs,
    callbacks=callbacks, 
    verbose=2
)

Epoch 1/4


I0000 00:00:1774113286.481669    2391 cuda_dnn.cc:529] Loaded cuDNN version 91002


2668/2668 - 351s - 132ms/step - accuracy: 0.2932 - auc: 0.8250 - f1_score: 0.2317 - loss: 2.3768 - val_accuracy: 0.3752 - val_auc: 0.8725 - val_f1_score: 0.2970 - val_loss: 2.0956 - learning_rate: 9.5000e-04
Epoch 2/4
2668/2668 - 352s - 132ms/step - accuracy: 0.4488 - auc: 0.9016 - f1_score: 0.3912 - loss: 1.8560 - val_accuracy: 0.4430 - val_auc: 0.9056 - val_f1_score: 0.3918 - val_loss: 1.8321 - learning_rate: 9.0250e-04
Epoch 3/4
2668/2668 - 337s - 126ms/step - accuracy: 0.5168 - auc: 0.9251 - f1_score: 0.4698 - loss: 1.6215 - val_accuracy: 0.5120 - val_auc: 0.9219 - val_f1_score: 0.4624 - val_loss: 1.6426 - learning_rate: 8.5737e-04
Epoch 4/4
2668/2668 - 329s - 123ms/step - accuracy: 0.5551 - auc: 0.9376 - f1_score: 0.5135 - loss: 1.4879 - val_accuracy: 0.5281 - val_auc: 0.9285 - val_f1_score: 0.4873 - val_loss: 1.5804 - learning_rate: 8.1451e-04


In [19]:
_ = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    initial_epoch=4,
    callbacks=callbacks, 
    verbose=2
)

Epoch 5/8
2668/2668 - 333s - 125ms/step - accuracy: 0.5921 - auc: 0.9492 - f1_score: 0.5596 - loss: 1.3467 - val_accuracy: 0.5251 - val_auc: 0.9245 - val_f1_score: 0.5017 - val_loss: 1.6171 - learning_rate: 7.7378e-04
Epoch 6/8
2668/2668 - 337s - 126ms/step - accuracy: 0.6190 - auc: 0.9549 - f1_score: 0.5891 - loss: 1.2611 - val_accuracy: 0.5439 - val_auc: 0.9349 - val_f1_score: 0.5033 - val_loss: 1.5205 - learning_rate: 7.3509e-04
Epoch 7/8
2668/2668 - 319s - 120ms/step - accuracy: 0.6559 - auc: 0.9619 - f1_score: 0.6285 - loss: 1.1576 - val_accuracy: 0.5742 - val_auc: 0.9382 - val_f1_score: 0.5371 - val_loss: 1.4565 - learning_rate: 6.9834e-04
Epoch 8/8
2668/2668 - 341s - 128ms/step - accuracy: 0.6706 - auc: 0.9660 - f1_score: 0.6465 - loss: 1.0855 - val_accuracy: 0.5926 - val_auc: 0.9415 - val_f1_score: 0.5644 - val_loss: 1.3924 - learning_rate: 6.6342e-04
